In [ ]:
import torch

In [ ]:
### FIlling matrix diagonals

dim = 5

matrix = torch.zeros((dim, dim), dtype=torch.long)
values = torch.ones(dim-1, dtype=torch.long)
token_pos = torch.arange(0, dim-1)
matrix[token_pos+1, token_pos] = values

print(matrix)

In [ ]:
from torch._tensor import Tensor


seq_len = 6
head_dim = 4

token_pos = torch.arange(seq_len)
d_pairs = torch.arange(0, head_dim // 2).repeat_interleave(2)

thetas = torch.pow(10000, -2*d_pairs/head_dim)
# print(thetas)
# print(token_pos, d_pairs)

print(token_pos.shape, d_pairs.shape, thetas.shape)

# Compute the outer product of indices and thetas
cos_pos_d_matrix = torch.sin(torch.outer(token_pos, thetas))
sin_pos_d_matrix = torch.cos(torch.outer(token_pos, thetas))

print(cos_pos_d_matrix)
print(sin_pos_d_matrix)



tokens = torch.rand(seq_len, head_dim)

rotated_tokens = torch.zeros_like(tokens)
for i, token in enumerate[Tensor](tokens):
    rot_matrix  = torch.zeros((head_dim, head_dim))
    values = torch.ones(head_dim, dtype=torch.long)
    rot_matrix[values, values] = cos_pos_d_matrix[i]
    rot_matrix[values[:-1]+1, values[:-1]] = sin_pos_d_matrix[i]
    rot_matrix[values[:-1], values[:-1]+1] = -sin_pos_d_matrix[i]
    
    rotated_token = torch.matmul(rot_matrix, token)
    print(rotated_token.shape, rotated_tokens[i].shape, token.shape)
    rotated_tokens[i] = rotated_token


In [ ]:
#first crappy implementation

seq_len = 6
head_dim = 8

token_pos = torch.arange(seq_len)
d_pairs = torch.arange(0, head_dim // 2)

thetas = torch.pow(10000, -2*d_pairs/head_dim)

# Compute the outer product of indices and thetas
cos_pos_d_matrix = torch.cos(torch.outer(token_pos, thetas.repeat_interleave(2)))
sin_base = torch.sin(torch.outer(token_pos, thetas))
sin_pos_d_matrix = torch.zeros((seq_len, head_dim))
sin_pos_d_matrix[:, 0::2] = sin_base
sin_pos_d_matrix = sin_pos_d_matrix[:, :-1]

tokens = torch.ones(seq_len, head_dim)

rotated_tokens = torch.zeros_like(tokens)
for i, token in enumerate[Tensor](tokens):
    rot_matrix  = torch.zeros((head_dim, head_dim))
    values = torch.arange(0, head_dim, dtype=torch.long)
    rot_matrix[values, values] = cos_pos_d_matrix[i]
    rot_matrix[values[:-1]+1, values[:-1]] = sin_pos_d_matrix[i]
    rot_matrix[values[:-1], values[:-1]+1] = -sin_pos_d_matrix[i]
    rotated_token = torch.matmul(rot_matrix, token)
    rotated_tokens[i] = rotated_token

In [ ]:
#better implementation, tested for a single token, single head

seq_len = 6
head_dim = 8

token_pos = torch.arange(seq_len)
d_pairs = torch.arange(0, head_dim // 2)

thetas = torch.pow(10000, -2*d_pairs/head_dim)
# print(thetas)
# print(token_pos, d_pairs)

print(token_pos.shape, d_pairs.shape, thetas.shape)

# Compute the outer product of indices and thetas
cos_pos_d_matrix = torch.cos(torch.outer(token_pos, thetas))
sin_pos_d_matrix = torch.sin(torch.outer(token_pos, thetas))
print("cos matrix\n", cos_pos_d_matrix)
print("sin matrix\n", sin_pos_d_matrix)



tokens = torch.ones(seq_len, head_dim)
tokens_even = tokens[:, ::2]
tokens_odd = tokens[:, 1::2]
print(tokens_even.shape)

rot_even = cos_pos_d_matrix[2] * tokens_even - sin_pos_d_matrix[2] * tokens_odd
rot_odd = + sin_pos_d_matrix[2] * tokens_even + cos_pos_d_matrix[2] * tokens_odd
tokens[:,::2] = rot_even
tokens[:,1::2] = rot_odd

print(tokens)

In [ ]:
#batches a full input B x seq_len x emb_dim

seq_len = 6
batch_size = 16
head_dim = 8
embedding_dim = 1*head_dim

cos_pos_d_matrix = cos_pos_d_matrix[None, :, None, :]
sin_pos_d_matrix = sin_pos_d_matrix[None, :, None, :]

tokens = torch.ones(batch_size, seq_len, embedding_dim)
tokens = tokens.view((batch_size, seq_len, embedding_dim//head_dim, head_dim))

tokens_even = tokens[:, :, :, ::2]
tokens_odd = tokens[:, :, :, 1::2]
rot_even = cos_pos_d_matrix * tokens_even - sin_pos_d_matrix * tokens_odd
rot_odd = + sin_pos_d_matrix * tokens_even + cos_pos_d_matrix * tokens_odd
tokens[:, :, :, ::2] = rot_even
tokens[:, :, :,1::2] = rot_odd

tokens.view((batch_size, seq_len, embedding_dim))